# 모델 압축: 가지치기 + 양자화 — Digit Recognizer (PyTorch) — Colab

학습된 CNN을 **가지치기(pruning)** 와 **양자화(quantization)** 로 압축하고, 크기·정확도·속도를 비교하는 기본 예제입니다.

- 핵심 흐름: 기준 모델 학습 → **가중치 가지치기**(희소화) → **동적 양자화**(float32→int8) → 효율 비교.
- IOAI 2025 과제(**Pixel Efficiency**, 모델 압축/효율)의 기반 기법입니다.
- [Digit Recognizer](https://www.kaggle.com/c/digit-recognizer) 데이터를 사용하며 토큰이 **노트북에 내장**되어 바로 실행됩니다.

> ℹ️ 양자화(동적)는 **CPU** 에서 동작합니다(추론 비교는 CPU 기준). 학습은 GPU 권장.  
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부 공유 금지.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Digit Recognizer 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 준비 & 기준 CNN 학습

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
X = df.drop("label", axis=1).values.reshape(-1,1,28,28).astype("float32")/255.0
y = df["label"].values.astype("int64")
rng = np.random.RandomState(42); idx = rng.permutation(len(X)); cut=int(len(X)*0.9)
tr, va = idx[:cut], idx[cut:]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(TensorDataset(torch.from_numpy(X[tr]), torch.from_numpy(y[tr])), batch_size=128, shuffle=True)
Xva = torch.from_numpy(X[va]); yva = y[va]

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64*7*7,128), nn.ReLU(), nn.Linear(128,10))
    def forward(self, x): return self.classifier(self.features(x))

model = CNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3); crit = nn.CrossEntropyLoss()
for epoch in range(3):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
print("기준 CNN 학습 완료")

## 3. 평가 유틸 (정확도 / 모델 크기 / 추론 속도)

In [ ]:
import time, os

@torch.no_grad()
def accuracy(m, dev="cpu"):
    m.eval().to(dev)
    pred = []
    for i in range(0, len(Xva), 512):
        pred.append(m(Xva[i:i+512].to(dev)).argmax(1).cpu().numpy())
    return (np.concatenate(pred) == yva).mean()

def model_size_mb(m):
    path = os.path.join(WORK_DIR, "_tmp.pt"); torch.save(m.state_dict(), path)
    mb = os.path.getsize(path)/1e6; os.remove(path); return mb

@torch.no_grad()
def infer_time(m, dev="cpu", n=2000):
    m.eval().to(dev); x = Xva[:n].to(dev)
    t = time.time(); m(x); return (time.time()-t)*1000

base_acc = accuracy(model, "cpu"); base_mb = model_size_mb(model); base_ms = infer_time(model)
print(f"[기준]  acc {base_acc:.4f} | size {base_mb:.2f} MB | infer {base_ms:.0f} ms")

## 4. 가지치기 (Pruning)

전역 L1 비구조적 가지치기로 가중치의 50% 를 0 으로 만듭니다(희소화). 보통 소량 재학습으로 정확도를 회복합니다.

In [ ]:
import torch.nn.utils.prune as prune
import copy

pruned = copy.deepcopy(model).to(device)
params = [(m, "weight") for m in pruned.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=0.5)

# 가지치기 후 짧게 재학습(정확도 회복)
opt2 = torch.optim.Adam(pruned.parameters(), lr=5e-4)
pruned.train()
for xb, yb in train_loader:
    xb, yb = xb.to(device), yb.to(device)
    opt2.zero_grad(); crit(pruned(xb), yb).backward(); opt2.step()

for m, name in params: prune.remove(m, name)  # 마스크 영구 적용

zeros = sum(int((m.weight == 0).sum()) for m, _ in params)
total = sum(m.weight.numel() for m, _ in params)
print(f"[가지치기] sparsity {zeros/total:.1%} | acc {accuracy(pruned,'cpu'):.4f}")

## 5. 양자화 (Dynamic Quantization)

Linear 레이어 가중치를 float32 → int8 로 양자화합니다(CPU 추론).

In [ ]:
import torch.ao.quantization as tq

# 양자화 엔진 설정 (Colab=fbgemm, 일부 환경=qnnpack)
_engs = torch.backends.quantized.supported_engines
torch.backends.quantized.engine = "fbgemm" if "fbgemm" in _engs else ("qnnpack" if "qnnpack" in _engs else _engs[-1])

cpu_model = copy.deepcopy(model).to("cpu").eval()
quantized = tq.quantize_dynamic(cpu_model, {nn.Linear}, dtype=torch.qint8)

q_acc = accuracy(quantized, "cpu"); q_mb = model_size_mb(quantized); q_ms = infer_time(quantized, "cpu")
print(f"엔진: {torch.backends.quantized.engine}")
print(f"[양자화] acc {q_acc:.4f} | size {q_mb:.2f} MB | infer {q_ms:.0f} ms")

## 6. 비교 결과

In [ ]:
import pandas as pd
pruned_zeros = zeros/total
table = pd.DataFrame([
    {"모델":"기준(float32)", "정확도":round(base_acc,4), "크기(MB)":round(base_mb,2), "추론(ms)":round(base_ms,0), "희소도":"0%"},
    {"모델":"가지치기",      "정확도":round(accuracy(pruned,"cpu"),4), "크기(MB)":round(model_size_mb(pruned),2), "추론(ms)":round(infer_time(pruned),0), "희소도":f"{pruned_zeros:.0%}"},
    {"모델":"양자화(int8)",  "정확도":round(q_acc,4), "크기(MB)":round(q_mb,2), "추론(ms)":round(q_ms,0), "희소도":"-"},
])
print(table.to_string(index=False))
table

## 🏁 제출 안내

이 노트북은 **모델 압축(가지치기/양자화)** 데모라 제출 리더보드가 없습니다.

- 데이터 출처: **[Digit Recognizer](https://www.kaggle.com/c/digit-recognizer)**

> IOAI 2025 *Pixel Efficiency*(모델 효율/압축) 과제의 기반 기법입니다. 가지치기는 희소도↑(희소 저장/추론 가속 기반), 양자화는 모델 크기↓ 와 CPU 추론 가속을 제공합니다.